# Exploring the synthetic forgery dataset

Companion notebook for the pipeline in `src/`:

1. Dataset composition (splits, forgery types, difficulty)
2. Visual audit: genuine vs the 4 forgery types + masks
3. Phone-capture augmentation samples
4. Training history of the baseline run

In [ ]:
%matplotlib inline
import json, os, sys
import cv2, numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))
from src.dataset import load_manifest, composition_table, filter_split

DATA = os.path.abspath('../data')
recs = load_manifest(os.path.join(DATA, 'manifest.jsonl'))
print(f'{len(recs)} records')
print(composition_table(recs))

In [ ]:
# Visual audit: one tampered image per forgery type x difficulty + its mask
types = ['copy_move', 'splicing', 'resampling', 'jpeg_recompression']
diffs = ['subtle', 'moderate', 'aggressive']
test = filter_split(recs, 'test')

fig, axes = plt.subplots(len(types), len(diffs) * 2, figsize=(15, 4 * len(types)))
for r_i, ft in enumerate(types):
    for c_i, d in enumerate(diffs):
        cand = [r for r in test if r['forgery_type'] == ft and r['difficulty'] == d]
        if not cand:
            continue
        r = cand[0]
        img = cv2.cvtColor(cv2.imread(os.path.join(DATA, r['image'])), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(os.path.join(DATA, r['mask']), 0)
        axes[r_i, c_i * 2].imshow(img)
        axes[r_i, c_i * 2].set_title(f'{ft}\n{d}', fontsize=9)
        axes[r_i, c_i * 2 + 1].imshow(mask, cmap='gray')
        axes[r_i, c_i * 2 + 1].set_title('mask', fontsize=9)
        for ax in (axes[r_i, c_i * 2], axes[r_i, c_i * 2 + 1]):
            ax.axis('off')
fig.tight_layout()

In [ ]:
# Phone-capture augmentation samples (train-time only)
from src.augmentations import capture_pipeline

aug = capture_pipeline()
base = cv2.imread(os.path.join(DATA, test[1]['image']))
fig, axes = plt.subplots(1, 5, figsize=(16, 3.4))
axes[0].imshow(cv2.cvtColor(base, cv2.COLOR_BGR2RGB)); axes[0].set_title('clean')
for ax in axes[1:]:
    out = aug(image=base)['image']
    ax.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
    ax.set_title('capture aug')
for ax in axes: ax.axis('off')
fig.tight_layout()

In [ ]:
# Training history of the baseline run (if it exists)
import pandas as pd

hist_path = os.path.abspath('../runs/baseline-ce/history.jsonl')
if os.path.exists(hist_path):
    df = pd.read_json(hist_path, lines=True)
    display(df)
    ax = df.plot(x='epoch', y=['val_acc', 'val_f1'], figsize=(8, 4))
    ax2 = df.plot(x='epoch', y=['val_far', 'val_far_subtle'], figsize=(8, 4))
    ax.grid(alpha=0.3); ax2.grid(alpha=0.3)
else:
    print('no baseline history yet - run scripts/run_experiments.sh first')